In [1]:
import torch
import subprocess
import os
import shutil
from gradio_client import Client, handle_file
import whisper
import shlex
import subprocess
import demucs.separate
from df.enhance import enhance, init_df, load_audio, save_audio
import torch
import gc
from transformers import AutoTokenizer
#from gptqmodel import GPTQModel
import whisper
import json
import os
from funasr import AutoModel
from funasr.utils.postprocess_utils import rich_transcription_postprocess
from difflib import SequenceMatcher
import re

def check_gpu():
    """Check if GPU is available with PyTorch."""
    print("Checking GPU availability with PyTorch...")
    if torch.cuda.is_available():
        print(f"GPU is available: {torch.cuda.get_device_name(0)}")
        return True
    else:
        print("GPU is not available.")
        return False

def check_whisper_installed():
    """Check if Whisper is installed."""
    print("Checking Whisper installation...")
    try:
        import whisper
        print("Whisper is installed.")
        return True
    except ImportError:
        print("Whisper is not installed.")
        return False

def check_ffmpeg():
    """Check if ffmpeg is installed."""
    print("Checking ffmpeg installation...")
    try:
        result = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
        if "ffmpeg version" in result.stdout:
            print("ffmpeg is installed.")
            return True
        else:
            print("ffmpeg is not installed.")
            return False
    except FileNotFoundError:
        print("ffmpeg is not installed.")
        return False



ModuleNotFoundError: No module named 'whisper'

In [ ]:
"""
Here we take the Raw audio(video) file and prepare the audio for the rest to come
"""
def separate_vocals(input_audio_path, output_dir, model='mdx_extra'):
    """
    Separate vocals and no-vocals with Demucs and save to custom output_dir.
    """
        
    print(f"Separating vocals from {input_audio_path}...")

    # Run Demucs CLI separation
    cmd = f'--two-stems vocals -n {model} "{input_audio_path}"'
    demucs.separate.main(shlex.split(cmd))

    base_name = os.path.splitext(os.path.basename(input_audio_path))[0]
    demucs_output_dir = os.path.join("separated", model, base_name)

    vocals_src = os.path.join(demucs_output_dir, "vocals.wav")
    no_vocals_src = os.path.join(demucs_output_dir, "no_vocals.wav")

    os.makedirs(output_dir, exist_ok=True)

    vocals_dst = os.path.join(output_dir, f"{base_name}_vocals.wav")
    no_vocals_dst = os.path.join(output_dir, f"{base_name}_no_vocals.wav")

    shutil.copy2(vocals_src, vocals_dst)
    shutil.copy2(no_vocals_src, no_vocals_dst)

    print(f"Vocals saved to: {vocals_dst}")
    print(f"No vocals saved to: {no_vocals_dst}")

    return vocals_dst, no_vocals_dst  # Return paths of saved files


def convert_to_48k_wav(input_audio_path, output_wav_path):
    """
    Convert any audio input to 48kHz stereo WAV using ffmpeg.
    """
    print(f"Converting {input_audio_path} to 48kHz WAV at {output_wav_path}...")
    subprocess.run([
        'ffmpeg', '-y', '-i', input_audio_path,
        '-ar', '48000',
        '-ac', '2',
        output_wav_path
    ], check=True)
    print("Conversion done.")


def denoise_audio(input_wav_path, output_wav_path):
    """
    Denoise/enhance the WAV audio using DeepFilterNet.
    """
    print(f"Denoising audio {input_wav_path} to {output_wav_path}...")
    model, df_state, _ = init_df()
    audio, _ = load_audio(input_wav_path, sr=df_state.sr())
    enhanced = enhance(model, df_state, audio)
    save_audio(output_wav_path, enhanced, df_state.sr())
    print("Denoising done.")


In [ ]:
class AudioTranscriber:
    def __init__(self, whisper_model_size="base", sensevoice_model_dir="iic/SenseVoiceSmall"):
        """Initialize both Whisper and SenseVoice models"""
        self.whisper_model = self.load_whisper_model(whisper_model_size)
        self.sensevoice_model = self.load_sensevoice_model(sensevoice_model_dir)
        
    def load_whisper_model(self, model_size="base"):
        """Load the Whisper model."""
        print(f"Loading Whisper {model_size} model...")
        model = whisper.load_model(model_size)
        print(f"Whisper {model_size} model loaded successfully.")
        return model

    def load_sensevoice_model(self, model_dir):
        """Load the SenseVoice model."""
        print("Loading SenseVoice model...")
        model = AutoModel(
            model=model_dir,
            trust_remote_code=True,
            remote_code="./model.py",    
            vad_model="fsmn-vad",
            vad_kwargs={"max_single_segment_time": 30000},
            device="cuda:0",
            disable_update=True,
        )
        print("SenseVoice model loaded successfully.")
        return model

    def transcribe_with_whisper(self, audio_file_path, language=None):
        """Transcribe audio using Whisper with detailed segments."""
        print(f"Transcribing audio file with Whisper: {audio_file_path}")
        
        options = {"word_timestamps": True}
        if language:
            options["language"] = language
        
        result = self.whisper_model.transcribe(audio_file_path, **options)
        print("Whisper transcription complete.")
        return result

    def extract_emotions_with_sensevoice(self, audio_file_path):
        """Extract emotions and transcription using SenseVoice."""
        print("Processing audio with SenseVoice for emotion detection...")
        
        if not os.path.exists(audio_file_path):
            raise FileNotFoundError(f"Input audio file not found: {audio_file_path}")

        try:
            res = self.sensevoice_model.generate(
                input=audio_file_path,
                cache={},
                language="auto",
                use_itn=True,
                batch_size_s=60,
                merge_vad=True,
                merge_length_s=15,
            )
        except Exception as e:
            print(f"Error during SenseVoice processing: {e}")
            raise

        text = rich_transcription_postprocess(res[0]["text"])
        print("SenseVoice processing complete.")
        return text

    def replace_emojis_with_tags(self, text):
        """Replace emojis with emotion tags."""
        emoji_to_tag = {
            "😊": "happy",
            "😡": "angry", 
            "😔": "sad",
            "😀": "laughter",
            "🎼": "music",
            "👏": "applause",
            "🤧": "cough & sneeze",
            "😭": "cry",
        }

        for emoji, tag in emoji_to_tag.items():
            text = text.replace(emoji, f"[{tag}]")

        return text.strip()

    def extract_emotions_from_text(self, text):
        """Extract emotion tags and their positions from text."""
        emotion_pattern = r'\[([^\]]+)\]'
        emotions = []
        
        for match in re.finditer(emotion_pattern, text):
            emotions.append({
                'emotion': match.group(1),
                'start_pos': match.start(),
                'end_pos': match.end(),
                'text_before': text[:match.start()],
                'text_after': text[match.end():]
            })
        
        # Remove emotion tags to get clean text for matching
        clean_text = re.sub(emotion_pattern, '', text).strip()
        clean_text = ' '.join(clean_text.split())  # Normalize whitespace
        
        return emotions, clean_text

    def fuzzy_match_segments(self, whisper_segments, sensevoice_text, emotions, threshold=0.6):
        """Match Whisper segments with SenseVoice emotions using fuzzy matching."""
        _, clean_sensevoice_text = self.extract_emotions_from_text(sensevoice_text)
        
        enhanced_segments = []
        
        for segment in whisper_segments:
            segment_text = segment['text'].strip()
            segment_copy = segment.copy()
            segment_copy['emotions'] = []
            
            # Find the best match in the SenseVoice text
            best_match_ratio = 0
            best_match_emotions = []
            
            # Try to find fuzzy matches for this segment
            for emotion in emotions:
                # Get text around the emotion for context
                emotion_context_start = max(0, emotion['start_pos'] - 50)
                emotion_context_end = min(len(sensevoice_text), emotion['end_pos'] + 50)
                emotion_context = sensevoice_text[emotion_context_start:emotion_context_end]
                
                # Remove emotion tags from context for comparison
                clean_context = re.sub(r'\[[^\]]+\]', '', emotion_context).strip()
                clean_context = ' '.join(clean_context.split())
                
                # Calculate similarity
                similarity = SequenceMatcher(None, segment_text.lower(), clean_context.lower()).ratio()
                
                if similarity > threshold and similarity > best_match_ratio:
                    best_match_ratio = similarity
                    best_match_emotions = [emotion['emotion']]
                elif similarity > threshold and abs(similarity - best_match_ratio) < 0.1:
                    # If similarity is close to best, add this emotion too
                    if emotion['emotion'] not in best_match_emotions:
                        best_match_emotions.append(emotion['emotion'])
            
            segment_copy['emotions'] = best_match_emotions
            segment_copy['emotion_confidence'] = best_match_ratio
            enhanced_segments.append(segment_copy)
        
        return enhanced_segments

    def process_audio(self, audio_file_path, output_dir, language=None):
        """Process audio with both models and create enhanced transcription."""
        
        # Ensure output directory exists
        os.makedirs(output_dir, exist_ok=True)
        
        # Get Whisper transcription
        whisper_result = self.transcribe_with_whisper(audio_file_path, language)
        
        # Get SenseVoice emotions
        sensevoice_text = self.extract_emotions_with_sensevoice(audio_file_path)
        sensevoice_tagged = self.replace_emojis_with_tags(sensevoice_text)
        
        # Extract emotions and clean text
        emotions, clean_sensevoice_text = self.extract_emotions_from_text(sensevoice_tagged)
        
        # Match segments with emotions
        enhanced_segments = self.fuzzy_match_segments(
            whisper_result['segments'], 
            sensevoice_tagged, 
            emotions
        )
        
        # Create enhanced result
        enhanced_result = {
            'text': whisper_result['text'],
            'language': whisper_result.get('language', 'unknown'),
            'segments': enhanced_segments,
            'sensevoice_raw': sensevoice_text,
            'sensevoice_tagged': sensevoice_tagged,
            'detected_emotions': [emotion['emotion'] for emotion in emotions]
        }
        
        # Save results
        self.save_results(enhanced_result, output_dir, audio_file_path)
        
        return enhanced_result

    def save_results(self, enhanced_result, output_dir, audio_file_path):
        """Save all results to files."""
        filename_base = os.path.splitext(os.path.basename(audio_file_path))[0]
        
        # Save enhanced JSON with emotions
        json_path = os.path.join(output_dir, f"{filename_base}_enhanced.json")
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(enhanced_result, f, indent=2, ensure_ascii=False)
        print(f"✅ Enhanced JSON transcript saved to: {json_path}")
        
        # Save readable transcript with emotions
        txt_path = os.path.join(output_dir, f"{filename_base}_with_emotions.txt")
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write("TRANSCRIPT WITH EMOTIONS\n")
            f.write("="*50 + "\n\n")
            
            for segment in enhanced_result['segments']:
                timestamp = f"[{segment['start']:.1f}s - {segment['end']:.1f}s]"
                emotions_str = f" {segment['emotions']}" if segment['emotions'] else ""
                f.write(f"{timestamp}{emotions_str}: {segment['text']}\n")
            
            f.write(f"\n\nDetected Emotions: {', '.join(enhanced_result['detected_emotions'])}")
        
        print(f"✅ Readable transcript saved to: {txt_path}")
        
        # Save SenseVoice output
        sensevoice_path = os.path.join(output_dir, f"{filename_base}_sensevoice.txt")
        with open(sensevoice_path, "w", encoding="utf-8") as f:
            f.write(enhanced_result['sensevoice_tagged'])
        print(f"✅ SenseVoice output saved to: {sensevoice_path}")

    def write_transcription_to_json(self, result, output_dir, filename_base):
        """Write the full transcription result to a JSON file."""
        output_path = os.path.join(output_dir, f"{filename_base}.json")
        
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
            
        print(f"✅ JSON transcript saved to: {output_path}")

In [ ]:
def translate_transcribed_text(model_id, input_file_path, output_dir):
    """
    Translates transcribed text using a quantized LLM (e.g., GPTQ).
    
    Args:
        model_id: Hugging Face model ID (e.g., "MaziyarPanahi/Meta-Llama-3-8B-Instruct-GPTQ")
        input_file_path: Path to the transcribed .txt file
        output_dir: Directory to save the translated file
        
    Returns:
        Path to the saved translated file
    """
    # Clear memory
    try: del model
    except: pass
    try: del tokenizer
    except: pass
    torch.cuda.empty_cache()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()

    # Load model and tokenizer
    model = GPTQModel.from_quantized(
        model_id,
        device="cuda:0",
        trust_remote_code=True
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    # Read original text
    with open(input_file_path, "r", encoding="utf-8") as file:
        original_text = file.read()

    # Prepare prompt
    messages = [
        {"role": "system", "content": "You are a translation chatbot that translates text from german to english! Only translate the given text."},
        {"role": "user", "content": original_text},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    # Generate response muessen tokens dann nochmal anpassen 
    outputs = model.generate(
        input_ids,
        do_sample=True,
        temperature=0.6,
        max_new_tokens=400,
        top_p=0.9,
    )

    # Decode and save result
    response = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)
    translated_text = response.split("assistant")[0].strip()
    # Prepare output path
    base_filename = os.path.splitext(os.path.basename(input_file_path))[0]
    translated_filename = f"{base_filename}_translated.txt"
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, translated_filename)

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(translated_text)

    print(f"Translated text saved to {output_path}")
    return output_path

In [2]:
def synthesize_speech(text, reference_audio_path, reference_text="", output_dir="../output"):
    """Synthesize speech using Fish Speech.
    
    Args:
        text: Text to synthesize
        reference_audio_path: Path to reference audio file
        reference_text: Optional reference text
        output_dir: Directory to save the output file
        
    Returns:
        Path to the generated audio file
    """
    print("Synthesizing speech with Fish Speech...")
    try:
        client = Client("http://fish-speech:7860/")
        
        result = client.predict(
            text=text,
            reference_id="",
            reference_audio=handle_file(reference_audio_path),
            reference_text=reference_text,
            max_new_tokens=0,
            chunk_length=200,
            top_p=0.5,
            repetition_penalty=1.7,
            temperature=0.6,
            seed=0,
            use_memory_cache="on",
            api_name="/partial"
        )
        
        # Handle the result
        if isinstance(result, tuple) and len(result) > 0:
            if isinstance(result[0], str) and result[0].endswith(".wav"):
                wav_file_path = result[0]
                
                # Create output directory if it doesn't exist
                os.makedirs(output_dir, exist_ok=True)
                
                # Save directly to output folder with original filename
                output_file_path = os.path.join(output_dir, os.path.basename(wav_file_path))
                shutil.move(wav_file_path, output_file_path)
                print(f"Generated .wav file saved to {output_file_path}")
                return output_file_path
            else:
                print(f"Unexpected result format: {result}")
                return None
        else:
            print(f"Unexpected result format: {result}")
            return None
    except Exception as e:
        print(f"Error in speech synthesis: {e}")
        return None


In [3]:
def process_audio_file(audio_file_path, reference_audio_path=None, whisper_model_size="base", language=None, output_dir="../output", model_id="MaziyarPanahi/Meta-Llama-3-8B-Instruct-GPTQ"):
    """Process an audio file: enhanced transcribe with emotions and then synthesize.
    
    Args:
        audio_file_path: Path to the audio file to transcribe
        reference_audio_path: Path to reference audio for synthesis (defaults to input audio if None)
        whisper_model_size: Size of the Whisper model to use
        language: Optional language code for transcription
        output_dir: Directory to save output files
        model_id: Model ID for translation
    
    Returns:
        Tuple of (path to generated audio file, path to transcription file, transcribed text, enhanced_result)
    """
    print(f"🎯 Processing audio file: {audio_file_path}")
    
    # Initialize the enhanced transcriber
    try:
        transcriber = AudioTranscriber(
            whisper_model_size=whisper_model_size,
            sensevoice_model_dir="iic/SenseVoiceSmall"
        )
    except Exception as e:
        print(f"❌ Failed to initialize enhanced transcriber: {e}")
        raise
    
    # Use the same audio file as reference if none provided
    if reference_audio_path is None:
        reference_audio_path = audio_file_path
    
    # Get base filename without extension for consistent naming
    base_filename = os.path.splitext(os.path.basename(audio_file_path))[0]
    
    # Create enhanced transcription subdirectory
    enhanced_output_dir = os.path.join(output_dir, "enhanced_transcription")
    
    # Process audio with enhanced transcription (includes emotion detection)
    print("🎭 Running enhanced transcription with emotion detection...")
    try:
        enhanced_result = transcriber.process_audio(
            audio_file_path=audio_file_path,
            output_dir=enhanced_output_dir,
            language=language
        )
        
        # Extract the clean transcribed text for synthesis
        transcribed_text = enhanced_result['text']
        
        print(f"✅ Enhanced transcription completed!")
        print(f"🎭 Detected emotions: {', '.join(enhanced_result['detected_emotions'])}")
        
    except Exception as e:
        print(f"❌ Enhanced transcription failed: {e}")
        raise
    
    # Save basic transcribed text to main output directory (for compatibility)
    text_file_path = save_transcribed_text(transcribed_text, output_dir, base_filename)
    
    # Save enhanced text with emotions for reference
    enhanced_text_path = save_enhanced_transcription_text(enhanced_result, output_dir, base_filename)
    
    # Translate described text and save text to file 
    print("🌐 Translating transcribed text...")
    try:
        translated_path = translate_transcribed_text(model_id, text_file_path, output_dir)
        print(f"✅ Translation saved to: {translated_path}")
    except Exception as e:
        print(f"❌ Translation failed: {e}")
        translated_path = None
    
    # Synthesize speech using the transcribed text
    print("🎤 Synthesizing speech from transcription...")
    try:
        output_audio_path = synthesize_speech(
            text=transcribed_text, 
            reference_audio_path=reference_audio_path, 
            output_dir=output_dir
        )
        print(f"✅ Speech synthesis completed: {output_audio_path}")
    except Exception as e:
        print(f"❌ Speech synthesis failed: {e}")
        output_audio_path = None
    
    return output_audio_path, text_file_path, transcribed_text, enhanced_result

def save_transcribed_text(text, output_dir, filename_base):
    """Save transcribed text to a file.
    
    Args:
        text: Transcribed text to save
        output_dir: Directory to save the text file
        filename_base: Base filename (without extension)
        
    Returns:
        Path to the saved text file
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Create output file path
    output_file_path = os.path.join(output_dir, f"{filename_base}.txt")
    
    # Write text to file
    with open(output_file_path, 'w', encoding='utf-8') as f:
        f.write(text)
    
    print(f"Transcribed text saved to {output_file_path}")
    return output_file_path


def save_enhanced_transcription_text(enhanced_result, output_dir, filename_base):
    """Save enhanced transcription text with emotions in a readable format."""
    os.makedirs(output_dir, exist_ok=True)
    
    enhanced_text_path = os.path.join(output_dir, f"{filename_base}_enhanced_readable.txt")
    
    with open(enhanced_text_path, 'w', encoding='utf-8') as f:
        f.write("ENHANCED TRANSCRIPTION WITH EMOTIONS\n")
        f.write("=" * 50 + "\n\n")
        
        f.write(f"Full Text: {enhanced_result['text']}\n\n")
        f.write(f"Language: {enhanced_result.get('language', 'unknown')}\n")
        f.write(f"Detected Emotions: {', '.join(enhanced_result['detected_emotions'])}\n\n")
        
        f.write("DETAILED SEGMENTS:\n")
        f.write("-" * 30 + "\n")
        
        for i, segment in enumerate(enhanced_result['segments'], 1):
            timestamp = f"[{segment['start']:.1f}s - {segment['end']:.1f}s]"
            emotions_str = f" 🎭[{', '.join(segment['emotions'])}]" if segment['emotions'] else ""
            confidence = f" (confidence: {segment.get('emotion_confidence', 0):.2f})" if segment['emotions'] else ""
            
            f.write(f"{i:3d}. {timestamp}{emotions_str}{confidence}\n")
            f.write(f"     \"{segment['text']}\"\n\n")
    
    print(f"📝 Enhanced readable transcription saved to: {enhanced_text_path}")
    return enhanced_text_path

# Optional: Enhanced synthesis function that can use emotion information
def synthesize_speech_with_emotions(enhanced_result, reference_audio_path, output_dir="../output"):
    """
    Enhanced speech synthesis that could potentially use emotion information.
    For now, uses the clean text, but structure allows for future emotion-aware synthesis.
    """
    print("🎤 Synthesizing speech with emotion awareness...")
    
    # Extract clean text
    clean_text = enhanced_result['text']
    
    # Future enhancement: Could modify synthesis based on detected emotions
    # For now, use standard synthesis
    output_audio_path = synthesize_speech(
        text=clean_text,
        reference_audio_path=reference_audio_path,
        output_dir=output_dir
    )
    
    # Could also create emotion-annotated audio segments in the future
    return output_audio_path


In [4]:
def main():
    # Check requirements
    has_gpu = check_gpu()
    has_whisper = check_whisper_installed()
    has_ffmpeg = check_ffmpeg()
    
    if not (has_whisper and has_ffmpeg):
        print("Missing required components. Please install them before continuing.")
        return
    
    # Define paths
    input_raw_audio = '../input/test_de.mp3'
    custom_folder = '../output/test-pipeline'
    converted_wav = os.path.join(custom_folder, 'vocals_48k.wav')
    denoised_wav = os.path.join(custom_folder, 'vocals_denoised.wav')
    input_audio = "../input/test_de.mp3"
    reference_audio = "../input/test_de.mp3"
    output_dir = "../output/test-pipeline"
    
    print("="*60)
    print("STARTING ENHANCED AUDIO PROCESSING PIPELINE")
    print("="*60)
    
    # Step 1: Separate vocals and no-vocals
    print("\n🎵 Step 1: Separating vocals from background...")
    vocals_path, no_vocals_path = separate_vocals(input_audio, custom_folder)
    print(f"✅ Vocals extracted to: {vocals_path}")

    # Step 2: Convert vocals to 48kHz WAV
    print("\n🔄 Step 2: Converting to 48kHz WAV...")
    convert_to_48k_wav(vocals_path, converted_wav)
    print(f"✅ Audio converted to: {converted_wav}")

    # Step 3: Denoise the converted WAV
    print("\n🎚️  Step 3: Denoising audio...")
    denoise_audio(converted_wav, denoised_wav)
    print(f"✅ Audio denoised to: {denoised_wav}")
    
    # Step 4: Enhanced transcription and synthesis pipeline
    print("\n🎭 Step 4: Running enhanced transcription and synthesis pipeline...")
    try:
        # Use the best quality audio available
        best_audio = denoised_wav if os.path.exists(denoised_wav) else converted_wav
        if not os.path.exists(best_audio):
            best_audio = input_audio
            
        output_audio_path, text_file_path, transcribed_text, enhanced_result = process_audio_file(
            audio_file_path=best_audio,
            reference_audio_path=reference_audio,
            whisper_model_size="base",
            language="",  # Set to the language of the audio or None for auto-detection
            output_dir=output_dir
        )
        
        if output_audio_path and text_file_path:
            print(f"✅ Enhanced pipeline completed successfully!")
            print(f"📄 Transcribed text preview: {transcribed_text[:100]}...")
            print(f"🔊 Generated audio saved to: {output_audio_path}")
            print(f"📝 Transcription saved to: {text_file_path}")
            print(f"🎭 Emotional segments: {len([s for s in enhanced_result['segments'] if s['emotions']])}")
        else:
            print("❌ Pipeline failed to complete successfully.")
            return
            
    except Exception as e:
        print(f"❌ Error in enhanced pipeline: {e}")
        return
    
    # Step 5: Generate summary report
    print("\n📋 PIPELINE COMPLETION SUMMARY")
    print("="*60)
    
    # Display emotion highlights
    emotion_segments = [s for s in enhanced_result['segments'] if s['emotions']]
    if emotion_segments:
        print(f"\n🌟 Emotional highlights found:")
        for i, segment in enumerate(emotion_segments[:5]):
            timestamp = f"[{segment['start']:.1f}s-{segment['end']:.1f}s]"
            emotions = f"[{', '.join(segment['emotions'])}]"
            preview = segment['text'][:50] + "..." if len(segment['text']) > 50 else segment['text']
            print(f"   {i+1}. {timestamp} {emotions}: {preview}")
    
    print(f"\n🎉 Enhanced pipeline processing completed!")
    print(f"📁 All outputs saved in: {output_dir}")
    print(f"🎭 Detected emotions: {', '.join(set(enhanced_result['detected_emotions']))}")

if __name__ == "__main__":
    main()


NameError: name 'check_gpu' is not defined